# Spectral Image Reconstruction Demo

This notebook walks through the main ideas in the project in a more visual, step-by-step way.

- Build a graph from image pixels
- Compute the normalized graph Laplacian
- Reconstruct the image with a limited number of eigenvectors
- Use the spectral embedding for segmentation


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from main import (
    build_features,
    build_weight_matrix,
    compute_normalized_laplacian,
    load_image,
    reconstruct_image,
    segment_image,
    spectral_decomposition,
)


## 1. Load and preview the image

We resize the image to keep the graph small enough to experiment with easily.

In [ ]:
image_array, _ = load_image(PROJECT_ROOT / "data" / "test.jpg", size=32)
plt.figure(figsize=(4, 4))
plt.imshow(image_array)
plt.title("Original Image")
plt.axis("off")
plt.show()


## 2. Build graph features

Each pixel becomes a node. Its feature vector combines:

- normalized `(x, y)` position
- RGB color values


In [ ]:
features, pixels, height, width = build_features(
    image_array,
    spatial_weight=0.1,
    color_weight=1.0,
)

print("Feature matrix shape:", features.shape)
print("Pixel matrix shape:", pixels.shape)


## 3. Construct the pixel graph and Laplacian

We connect each pixel to its nearest neighbors, then build the normalized graph Laplacian.

In [ ]:
weight_matrix = build_weight_matrix(features, neighbors=15)
normalized_laplacian = compute_normalized_laplacian(weight_matrix)

print("Weight matrix shape:", weight_matrix.shape)
print("Normalized Laplacian shape:", normalized_laplacian.shape)


## 4. Reconstruct the image with different values of `k`

Smaller `k` keeps fewer graph frequency components, so the reconstruction is smoother. Larger `k` preserves more detail.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_array)
axes[0].set_title("Original")
axes[0].axis("off")

for axis, k in zip(axes[1:], [10, 50]):
    _, eigenvectors = spectral_decomposition(normalized_laplacian, eigen_count=k)
    reconstructed = reconstruct_image(pixels, eigenvectors).reshape(height, width, 3)
    axis.imshow(reconstructed)
    axis.set_title(f"Reconstruction k={eigenvectors.shape[1]}")
    axis.axis("off")

plt.tight_layout()
plt.show()


## 5. Use the eigenvectors as a spectral embedding for segmentation

Here we take a few eigenvectors, treat them as a new feature space for each pixel, and cluster them with k-means.

In [ ]:
_, embedding_eigenvectors = spectral_decomposition(normalized_laplacian, eigen_count=10)
embedding = embedding_eigenvectors[:, 1:min(4, embedding_eigenvectors.shape[1])]
segmented = segment_image(embedding, clusters=5, height=height, width=width)

plt.figure(figsize=(4, 4))
plt.imshow(segmented, cmap="viridis")
plt.title("Spectral Segmentation (5 clusters)")
plt.axis("off")
plt.show()


## 6. Takeaway

- For reconstruction, Laplacian eigenvectors act like graph frequency basis functions.
- For segmentation, they give a graph-aware embedding that groups related pixels together.
